# Running EMM

In [1]:
# This notebook compares Beam Search and BUSPaS for finding exceptional
# long-term weekly-sales trajectory subgroups in the Walmart dataset.

# Import libraries and define the shared search settings used in the main experiment.
import time
import itertools
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# Search settings: identical for both methods
D_MAX = 3       # maximum number of conditions in a subgroup
Q = 5           # return the 5 best subgroups
W = 20          # beam width for Beam Search
U = 150         # closest trajectory pairs used by BUSPaS
MIN_COV = 0.01  # subgroup must contain at least 1% of rows

# Load the preprocessed Walmart data and separate subgroup descriptors
# from the standardized weekly-sales trajectories.

df = pd.read_csv("final_walmart_data.csv")

descriptor_cols = [
    "Store", "Dept", "Type",
    "Temperature_mean_category", "Fuel_Price_mean_category",
    "CPI_mean_category", "Unemployment_mean_category",
    "Temperature_slope_category", "Fuel_Price_slope_category",
    "CPI_slope_category", "Unemployment_slope_category",
    "Size_category"
]

target_cols = [col for col in df.columns if col not in descriptor_cols]

# A: descriptors; Y: already-standardized weekly-sales trajectories
A = df[descriptor_cols].to_numpy(dtype=object)
Y = df[target_cols].to_numpy(dtype=float)

print("A shape:", A.shape)  # 2628 rows × 12 descriptors
print("Y shape:", Y.shape)  # 2628 rows × 141 weeks

A shape: (2628, 12)
Y shape: (2628, 143)


## Quality measure

In [2]:
# Define the distance-based subgroup quality measure and helper functions
# used by both search algorithms.

class Quality:
    def __init__(self, Y):
        self.D = cdist(Y, Y)
        self.rowsum = self.D.sum(1)
        self.cache = {}

    def __call__(self, m):
        key = m.tobytes()
        if key not in self.cache:
            n = m.sum()
            if n < max(2, MIN_COV * len(m)) or n == len(m):
                self.cache[key] = None
            else:
                s = self.D @ m
                within = s[m].sum()
                beta = within / (n * (n - 1))
                alpha = (self.rowsum[m].sum() - within) / (n * (len(m) - n))
                self.cache[key] = alpha / (beta + 1)
        return self.cache[key]

def mask_of(A, desc):
    mask = np.ones(len(A), dtype=bool)

    for column_index, value in desc:
        mask &= (A[:, column_index] == value)

    return mask

def all_conditions(A):
    # Creates conditions such as Type = A, Store = 1, CPI_mean_category = high
    return [
        (column_index, value)
        for column_index in range(A.shape[1])
        for value in pd.unique(A[:, column_index])
    ]

## Beam search

In [3]:
# Define Beam Search, which expands the best partial subgroup descriptions
# up to the maximum description length.

def beam_search(A, qual, w=W):
    conditions = all_conditions(A)
    results, seeds, seen = [], [()], set()

    for _ in range(D_MAX):
        beam = []

        for seed in seeds:
            used_columns = {column_index for column_index, _ in seed}

            for condition in conditions:
                if condition[0] in used_columns:
                    continue

                desc = tuple(sorted(seed + (condition,)))

                if desc in seen:
                    continue

                seen.add(desc)
                mask = mask_of(A, desc)
                score = qual(mask)

                if score is not None:
                    results.append((score, desc, mask))
                    beam.append((score, desc))

        beam.sort(key=lambda x: x[0], reverse=True)
        seeds = [desc for _, desc in beam[:w]]

    return sorted(results, key=lambda x: x[0], reverse=True)[:Q], len(seen)

## Bottom-up search

In [4]:
# Define BUSPaS, which generates subgroup candidates from pairs of
# similar sales trajectories and their shared descriptor values.

def bottom_up_search(A, qual):
    distances = qual.D
    upper_triangle = np.triu_indices(len(distances), 1)
    flat_distances = distances[upper_triangle]

    closest = np.argpartition(flat_distances, U)[:U]
    candidate_descriptions = set()

    for i, j in zip(upper_triangle[0][closest], upper_triangle[1][closest]):
        shared_conditions = [
            (column_index, A[i, column_index])
            for column_index in range(A.shape[1])
            if A[i, column_index] == A[j, column_index]
        ]

        for size in range(1, min(D_MAX, len(shared_conditions)) + 1):
            candidate_descriptions.update(
                itertools.combinations(shared_conditions, size)
            )

    results = []

    for desc in candidate_descriptions:
        mask = mask_of(A, desc)
        score = qual(mask)

        if score is not None:
            results.append((score, desc, mask))

    return sorted(results, key=lambda x: x[0], reverse=True)[:Q], len(candidate_descriptions)

## Experiment 

Both algorithms are run on the same Walmart descriptors and standardized weekly-sales trajectories using the same distance-based quality measure. This makes their subgroup quality, number of evaluated candidates, and runtime directly comparable.

The settings are:

- `D_MAX = 3`: a subgroup description may contain up to three conditions.
- `MIN_COV = 0.01`: a valid subgroup must contain at least 1% of all trajectories.
- `Q = 5`: retain the five highest-quality subgroups.
- `W = 20`: Beam Search keeps 20 promising partial descriptions at each search level.
- `U = 150`: BUSPaS generates candidates from the 150 closest trajectory pairs.

In [5]:
# Run both algorithms using the same data, quality measure, and subgroup
# constraints to compare their top subgroups and search cost fairly.

quality_bs = Quality(Y)

start = time.time()
top_bs, candidates_bs = beam_search(A, quality_bs)
time_bs = time.time() - start

quality_bus = Quality(Y)

start = time.time()
top_bus, candidates_bus = bottom_up_search(A, quality_bus)
time_bus = time.time() - start

## Results

This section converts the output of each algorithm into a readable table. For every returned subgroup, it shows:
- rank: position based on the distance-based quality score;
- description: the descriptor condition(s) defining the subgroup;
- quality_score: how internally similar and externally distinct its sales trajectories are;
- coverage_n: number of trajectories in the subgroup;
- coverage_pct: subgroup size as a percentage of the full dataset.
It also displays the total number of candidate subgroups evaluated and the runtime for each algorithm.
In this run, Beam Search and BUSPaS returned the same five highest-quality department subgroups. However, Beam Search evaluated fewer candidates (1,646 vs. 2,292) and ran faster (0.56 vs. 1.31 seconds).

In [6]:
def results_table(top):
    rows = []

    for rank, (score, desc, mask) in enumerate(top, start=1):
        description = " AND ".join(
            f"{descriptor_cols[column_index]} = {value}"
            for column_index, value in desc
        )

        rows.append({
            "rank": rank,
            "description": description,
            "quality_score": score,
            "coverage_n": int(mask.sum()),
            "coverage_pct": 100 * mask.mean()
        })

    return pd.DataFrame(rows)

print("Beam Search")
display(results_table(top_bs))
print(f"Candidates evaluated: {candidates_bs:,}")
print(f"Time: {time_bs:.2f} seconds")

print("\nBUSPaS")
display(results_table(top_bus))
print(f"Candidates evaluated: {candidates_bus:,}")
print(f"Time: {time_bus:.2f} seconds")

Beam Search


,rank,description,quality_score,coverage_n,coverage_pct
0,1,Dept = 55,2.523385,37,1.407915
1,2,Dept = 72,2.247452,38,1.445967
2,3,Dept = 59,2.233055,32,1.217656
3,4,Dept = 67,2.129413,45,1.712329
4,5,Dept = 5,2.104108,41,1.560122


Candidates evaluated: 1,646
Time: 0.26 seconds

BUSPaS


,rank,description,quality_score,coverage_n,coverage_pct
0,1,Dept = 55,2.523385,37,1.407915
1,2,Dept = 72,2.247452,38,1.445967
2,3,Dept = 59,2.233055,32,1.217656
3,4,Dept = 67,2.129413,45,1.712329
4,5,Dept = 5,2.104108,41,1.560122


Candidates evaluated: 2,292
Time: 0.74 seconds


## More experiments

In [7]:
# Create a compact summary of the best subgroup from each experimental run.

def top_summary(top):
    if len(top) == 0:
        return {
            "best_quality": np.nan,
            "best_coverage_n": np.nan,
            "best_description": None
        }

    score, desc, mask = top[0]

    description = " AND ".join(
        f"{descriptor_cols[column_index]} = {value}"
        for column_index, value in desc
    )

    return {
        "best_quality": score,
        "best_coverage_n": int(mask.sum()),
        "best_description": description
    }

In [8]:
# Compare the best subgroup found by each method as the number of
# available descriptors increases, while keeping all search settings fixed.

full_descriptor_sets = {
    "4 descriptors": [
        "Store", "Dept", "Type", "Size_category"
    ],
    "8 descriptors": [
        "Store", "Dept", "Type", "Size_category",
        "Temperature_mean_category",
        "Fuel_Price_mean_category",
        "CPI_mean_category",
        "Unemployment_mean_category"
    ],
    "12 descriptors": [
        "Store", "Dept", "Type", "Size_category",
        "Temperature_mean_category",
        "Fuel_Price_mean_category",
        "CPI_mean_category",
        "Unemployment_mean_category",
        "Temperature_slope_category",
        "Fuel_Price_slope_category",
        "CPI_slope_category",
        "Unemployment_slope_category"
    ]
}


def run_comparison(A_subset, cols, label):
    rows = []

    # Beam Search
    quality_bs = Quality(Y)
    start = time.time()
    top_bs, candidates_bs = beam_search(A_subset, quality_bs, w=W)
    runtime_bs = time.time() - start

    score_bs, desc_bs, mask_bs = top_bs[0]
    description_bs = " AND ".join(
        f"{cols[column_index]} = {value}"
        for column_index, value in desc_bs
    )

    rows.append({
        "method": "Beam Search",
        "descriptor_set": label,
        "n_descriptors": len(cols),
        "best_quality": score_bs,
        "best_coverage_n": int(mask_bs.sum()),
        "best_description": description_bs,
        "candidates_evaluated": candidates_bs,
        "runtime_seconds": runtime_bs
    })

    # BUSPaS
    quality_bus = Quality(Y)
    start = time.time()
    top_bus, candidates_bus = bottom_up_search(A_subset, quality_bus)
    runtime_bus = time.time() - start

    score_bus, desc_bus, mask_bus = top_bus[0]
    description_bus = " AND ".join(
        f"{cols[column_index]} = {value}"
        for column_index, value in desc_bus
    )

    rows.append({
        "method": "BUSPaS",
        "descriptor_set": label,
        "n_descriptors": len(cols),
        "best_quality": score_bus,
        "best_coverage_n": int(mask_bus.sum()),
        "best_description": description_bus,
        "candidates_evaluated": candidates_bus,
        "runtime_seconds": runtime_bus
    })

    return rows


full_dimension_rows = []

for label, cols in full_descriptor_sets.items():
    A_subset = df[cols].to_numpy(dtype=object)

    full_dimension_rows.extend(
        run_comparison(A_subset, cols, label)
    )

full_dimension_results = pd.DataFrame(full_dimension_rows)

display(
    full_dimension_results.sort_values(
        ["n_descriptors", "method"]
    ).reset_index(drop=True)
)

,method,descriptor_set,n_descriptors,best_quality,best_coverage_n,best_description,candidates_evaluated,runtime_seconds
0,BUSPaS,4 descriptors,4,2.523385,37,Dept = 55,68,0.107161
1,Beam Search,4 descriptors,4,2.523385,37,Dept = 55,1142,0.181863
2,BUSPaS,8 descriptors,8,2.523385,37,Dept = 55,665,0.276676
3,Beam Search,8 descriptors,8,2.523385,37,Dept = 55,1394,0.209103
4,BUSPaS,12 descriptors,12,2.523385,37,Dept = 55,2292,0.685517
5,Beam Search,12 descriptors,12,2.523385,37,Dept = 55,1646,0.267393


This experiment removes Store and Dept only for a diagnostic experiment, because Dept = 55 masks the effect of added descriptors.

Results
- Removing `Store` and `Dept` forced both algorithms to use combinations of contextual descriptors, rather than relying on a single department rule.

- Beam Search found a higher-quality subgroup than BUSPaS at all tested dimensionalities:

  - 4 descriptors: 1.119 vs. 1.012
  - 8 descriptors: 1.128 vs. 1.012
  - 10 descriptors: 1.119 vs. 1.012

- Beam Search’s subgroups were smaller (33–57 trajectories) than BUSPaS’s subgroup (65 trajectories), but all met the minimum 1% coverage requirement.

- BUSPaS’s best quality did not improve as more descriptors were added; it remained 1.012 for 4, 8, and 10 descriptors.

- BUSPaS evaluated substantially more candidates as dimensionality increased: 72 → 566 → 1,100.

- Beam Search’s candidate count also increased, but more gradually: 144 → 579 → 821.

- In this Walmart context-only analysis, Beam Search found more exceptional subgroups under the chosen settings, while BUSPaS became more computationally expensive without improving its best subgroup quality.

In [9]:
# Diagnostic analysis: remove Store and Dept to examine whether increasing
# contextual descriptors changes the subgroups found by the two methods.

context_descriptor_sets = {
    "4 context descriptors": [
        "Type",
        "Size_category",
        "Temperature_mean_category",
        "Fuel_Price_mean_category"
    ],
    "8 context descriptors": [
        "Type",
        "Size_category",
        "Temperature_mean_category",
        "Fuel_Price_mean_category",
        "CPI_mean_category",
        "Unemployment_mean_category",
        "Temperature_slope_category",
        "Fuel_Price_slope_category"
    ],
    "10 context descriptors": [
        "Type",
        "Size_category",
        "Temperature_mean_category",
        "Fuel_Price_mean_category",
        "CPI_mean_category",
        "Unemployment_mean_category",
        "Temperature_slope_category",
        "Fuel_Price_slope_category",
        "CPI_slope_category",
        "Unemployment_slope_category"
    ]
}

context_dimension_rows = []

for label, cols in context_descriptor_sets.items():
    A_subset = df[cols].to_numpy(dtype=object)

    context_dimension_rows.extend(
        run_comparison(A_subset, cols, label)
    )

context_dimension_results = pd.DataFrame(context_dimension_rows)

display(
    context_dimension_results.sort_values(
        ["n_descriptors", "method"]
    ).reset_index(drop=True)
)

,method,descriptor_set,n_descriptors,best_quality,best_coverage_n,best_description,candidates_evaluated,runtime_seconds
0,BUSPaS,4 context descriptors,4,1.012414,65,Size_category = high AND Temperature_mean_cate...,72,0.110835
1,Beam Search,4 context descriptors,4,1.119367,33,Type = A AND Size_category = low AND Fuel_Pric...,144,0.089362
2,BUSPaS,8 context descriptors,8,1.012414,65,Size_category = high AND Fuel_Price_slope_cate...,566,0.394674
3,Beam Search,8 context descriptors,8,1.127829,57,Size_category = low AND Temperature_slope_cate...,579,0.316979
4,BUSPaS,10 context descriptors,10,1.012414,65,Size_category = high AND Temperature_mean_cate...,1100,0.559167
5,Beam Search,10 context descriptors,10,1.119367,33,Temperature_mean_category = high AND Fuel_Pric...,821,0.322756


This section tests how the amount of search effort affects each algorithm’s results.
The subgroup constraints remain fixed:
- D_MAX = 3: descriptions may contain up to three conditions;
- MIN_COV = 0.01: each subgroup must contain at least 1% of trajectories;
- Q = 5: retain the five highest-quality subgroups.
For Beam Search, the code tests different beam widths (W = 10, 20, 50, 100). A larger beam width keeps more promising partial subgroup descriptions and therefore explores more candidates.
For BUSPaS, the code tests different numbers of closest trajectory pairs (U = 50, 150, 300, 500). A larger value generates candidates from more similar trajectory pairs.
For every setting, the table reports the best subgroup found, its quality and coverage, the number of candidates evaluated, and runtime. This shows whether increasing the search effort improves the result and how much additional computational cost it requires.

The results show how much search effort each algorithm needs before it finds the best subgroup.
- Beam Search found Dept = 55 with quality 2.523 even at its smallest tested width, W = 10.
- Increasing W to 20, 50, or 100 did not improve the best subgroup. It only increased candidates evaluated and runtime.
- BUSPaS, with only U = 50 closest pairs, found a weaker subgroup: Dept = 72, quality 2.247.
- Once U reached 150, BUSPaS found the same best subgroup as Beam Search: Dept = 55, quality 2.523.
- Increasing U further to 300 or 500 did not improve the result; it only increased search cost.

Key finding: On this Walmart dataset, Beam Search reaches the best subgroup with less search effort. BUSPaS needs at least 150 close trajectory pairs to reach the same result.

In [10]:
# Test how different search budgets affect the subgroup quality found
# and the computational cost of Beam Search and BUSPaS.

D_MAX = 3
MIN_COV = 0.01
Q = 5

beam_widths = [10, 20, 50, 100]
bus_pairs = [50, 150, 300, 500]

results = []

# Beam Search tests
for W in beam_widths:
    quality = Quality(Y)

    start = time.time()
    top, candidates = beam_search(A, quality, w=W)
    runtime = time.time() - start

    summary = top_summary(top)

    results.append({
        "method": "Beam Search",
        "parameter": "W",
        "value": W,
        "candidates_evaluated": candidates,
        "runtime_seconds": runtime,
        **summary
    })

# BUSPaS tests
for U in bus_pairs:
    quality = Quality(Y)

    start = time.time()
    top, candidates = bottom_up_search(A, quality)
    runtime = time.time() - start

    summary = top_summary(top)

    results.append({
        "method": "BUSPaS",
        "parameter": "U",
        "value": U,
        "candidates_evaluated": candidates,
        "runtime_seconds": runtime,
        **summary
    })

budget_results = pd.DataFrame(results)
display(budget_results)

,method,parameter,value,candidates_evaluated,runtime_seconds,best_quality,best_coverage_n,best_description
0,Beam Search,W,10,896,0.206717,2.523385,37,Dept = 55
1,Beam Search,W,20,1646,0.243136,2.523385,37,Dept = 55
2,Beam Search,W,50,5722,0.502272,2.523385,37,Dept = 55
3,Beam Search,W,100,15627,1.062374,2.523385,37,Dept = 55
4,BUSPaS,U,50,1282,0.428510,2.247452,38,Dept = 72
5,BUSPaS,U,150,2292,0.696350,2.523385,37,Dept = 55
6,BUSPaS,U,300,3016,0.788856,2.523385,37,Dept = 55
7,BUSPaS,U,500,3702,0.895279,2.523385,37,Dept = 55


This section tests whether the comparison between Beam Search and BUSPaS changes when different subgroup constraints are used.
The search budgets remain fixed:
- W = 20 for Beam Search;
- U = 150 for BUSPaS.
The code then tests combinations of:
- D_MAX = 1, 2, 3: the maximum number of conditions allowed in a subgroup description;
- MIN_COV = 0.01, 0.03, 0.05: the minimum percentage of trajectories that a valid subgroup must contain.
For each combination, both algorithms are run using the same quality measure. The results table reports the best subgroup found, its quality and coverage, the number of candidates evaluated, and runtime.
This analysis checks whether the algorithms still find similar results when subgroup descriptions are allowed to be more complex or when larger subgroup coverage is required.

When MIN_COV = 0.01, both algorithms consistently find the same best subgroup: Dept = 55, with a quality score of 2.523 and coverage of 37 trajectories. This result does not change when the maximum description length increases from one to three conditions.
When the minimum coverage is increased to 3% or 5%, the small Dept = 55 subgroup is no longer valid because it contains only 1.4% of trajectories. The algorithms must therefore search for broader subgroups with lower quality scores.
For simple descriptions (D_MAX = 1), both methods return the same larger subgroup, Temperature_mean_category = low, with coverage of 920 trajectories and quality of approximately 0.963.
For more complex descriptions (D_MAX = 2 or 3) and MIN_COV = 0.03, Beam Search and BUSPaS sometimes return different best subgroups. In particular, with D_MAX = 3 and MIN_COV = 0.03, Beam Search finds a subgroup with higher quality (1.048) than BUSPaS (1.003), although the subgroup sizes are also different.
Overall, the results show that the algorithms agree when simple or very small subgroups are allowed. Differences appear when larger and more complex subgroup descriptions are required.

In [11]:
W = 20
U = 150

depth_values = [1, 2, 3]
coverage_values = [0.01, 0.03, 0.05]

results = []

for D_MAX in depth_values:
    for MIN_COV in coverage_values:

        # Beam Search
        quality_bs = Quality(Y)
        start = time.time()
        top_bs, candidates_bs = beam_search(A, quality_bs, w=W)
        runtime_bs = time.time() - start

        results.append({
            "method": "Beam Search",
            "D_MAX": D_MAX,
            "MIN_COV": MIN_COV,
            "candidates_evaluated": candidates_bs,
            "runtime_seconds": runtime_bs,
            **top_summary(top_bs)
        })

        # BUSPaS
        quality_bus = Quality(Y)
        start = time.time()
        top_bus, candidates_bus = bottom_up_search(A, quality_bus)
        runtime_bus = time.time() - start

        results.append({
            "method": "BUSPaS",
            "D_MAX": D_MAX,
            "MIN_COV": MIN_COV,
            "candidates_evaluated": candidates_bus,
            "runtime_seconds": runtime_bus,
            **top_summary(top_bus)
        })

constraint_results = pd.DataFrame(results)
display(constraint_results)

,method,D_MAX,MIN_COV,candidates_evaluated,runtime_seconds,best_quality,best_coverage_n,best_description
0,Beam Search,1,0.01,146,0.137267,2.523385,37,Dept = 55
1,BUSPaS,1,0.01,40,0.087510,2.523385,37,Dept = 55
2,Beam Search,1,0.03,146,0.030981,0.963424,920,Temperature_mean_category = low
3,BUSPaS,1,0.03,40,0.070688,0.963424,920,Temperature_mean_category = low
4,Beam Search,1,0.05,146,0.034533,0.963424,920,Temperature_mean_category = low
5,BUSPaS,1,0.05,40,0.083030,0.963424,920,Temperature_mean_category = low
6,Beam Search,2,0.01,1646,0.227208,2.523385,37,Dept = 55
7,BUSPaS,2,0.01,500,0.295723,2.523385,37,Dept = 55
8,Beam Search,2,0.03,2827,0.413100,0.997795,80,Type = C AND Fuel_Price_mean_category = high
9,BUSPaS,2,0.03,500,0.309722,0.993732,251,Temperature_slope_category = medium AND Fuel_P...


# Summary


## What I did

- Used **2,628 Store–Department trajectories**, each with **143 standardized weekly-sales values**.
- Used 12 descriptors: Store, Department, Type, Size, and categorical mean/slope summaries for Temperature, Fuel Price, CPI, and Unemployment.
- Used the same distance-based trajectory quality measure for both methods.
- Compared the implemented BUSPaS-style bottom-up search and Beam Search under the same main constraints:

  - `D_MAX = 3`
  - `MIN_COV = 0.01`
  - `Q = 5`
  - `W = 20` for Beam Search
  - `U = 150` for BUSPaS

## Overall results

- With all 12 descriptors, both methods found the exact same top five subgroups: departments 55, 72, 59, 67, and 5.
- The best subgroup was `Dept = 55`, with quality **2.523** and 37 trajectories.
- Beam Search found these results more efficiently:

  - Beam Search: 1,646 candidates; 0.26 seconds
  - BUSPaS: 2,292 candidates; 0.74 seconds

- When increasing the full descriptor set from 4 to 8 to 12, the best subgroup did not change because `Dept = 55` dominated every run.
- In the context-only diagnostic, where Store and Department were removed, Beam Search found higher-quality subgroups at every tested descriptor count:

  - 4 descriptors: BS 1.119 vs. BUSPaS 1.012
  - 8 descriptors: BS 1.128 vs. BUSPaS 1.012
  - 10 descriptors: BS 1.119 vs. BUSPaS 1.012

- However, Beam Search’s subgroups were smaller (33–57 trajectories) than BUSPaS’s (65 trajectories). Both meet the minimum coverage requirement, but this should be mentioned when comparing quality.

- The budget experiment justifies the settings:

  - Beam Search already finds the best subgroup at `W = 10`.
  - BUSPaS needs at least `U = 150`; with `U = 50`, it misses `Dept = 55`.

## Main conclusion

For this Walmart case study, Beam Search either matches BUSPaS or finds a higher-quality contextual subgroup while generally evaluating fewer candidates. BUSPaS becomes more expensive as more descriptors are added, without improving its best contextual subgroup quality under the selected settings.
